In [ ]:
import os
import ast
import wfdb
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers, models, callbacks
import matplotlib.pyplot as plt
from sklearn.preprocessing import MultiLabelBinarizer

# ==========================================
# 1. LETTURA DATASET E PREPARAZIONE DATI (codice fornito direttamente dai creatori di PTB-XL)
# ==========================================

def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data


path = './dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'

sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))
# Load raw signal data
X = load_raw_data(Y, sampling_rate, path)

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

In [ ]:
def is_normal(superclasses):
    # Ritorna 1.0 se l'ECG è NORM e non ha altre patologie concorrenti
    if 'NORM' in superclasses and len(superclasses) == 1:
        return 1.0
    else:
        return 0.0 # È patologico

print("Creazione etichette binarie NORM vs ABNORM...")
Y_binary = Y['diagnostic_superclass'].apply(is_normal).values
Y_binary = Y_binary.astype(np.float32)

norm_count = np.sum(Y_binary == 1)
abnorm_count = np.sum(Y_binary == 0)
print(f"Tracciati Normali (1): {norm_count}")
print(f"Tracciati Anormali (0): {abnorm_count}")

# ==========================================
# 2. DIVISIONE TRA TRAIN, VALIDATION E TEST
# ==========================================
test_fold = 10
val_fold = 9

# Ora usiamo Y_binary per assegnare correttamente le etichette 0 e 1
X_train = X[(Y.strat_fold != test_fold) & (Y.strat_fold != val_fold)]
Y_train = Y_binary[(Y.strat_fold != test_fold) & (Y.strat_fold != val_fold)]

X_val = X[Y.strat_fold == val_fold]
Y_val = Y_binary[Y.strat_fold == val_fold]

X_test = X[Y.strat_fold == test_fold]
Y_test = Y_binary[Y.strat_fold == test_fold]

# ==========================================
# 3. ESTRAZIONE D1, CONVERSIONE E NORMALIZZAZIONE
# ==========================================
# Estrazione singola derivazione D1 (indice 0) e conversione a float32 per TF
X_train = X_train[:, :, 0:1].astype(np.float32)
X_val = X_val[:, :, 0:1].astype(np.float32)
X_test = X_test[:, :, 0:1].astype(np.float32)

# Normalizzazione Z-score (calcolata SOLO sul training set per evitare data leakage)
mean = np.mean(X_train)
std = np.std(X_train)

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

In [ ]:
print(f"train: {X_train.shape} - {Y_train.shape[0]}")
print(f"val: {X_val.shape} - {Y_val.shape[0]}")
print(f"test: {X_test.shape} - {Y_test.shape[0]}")


In [ ]:
import numpy as np

def segmenta_dataset(X, y):
    """
    X: array di shape (N, 1000, 1)
    y: array delle label di shape (N,) oppure (N, num_classi)
    """
    # 1. Estraiamo le 3 finestre sovrapposte usando lo slicing (istantaneo su NumPy)
    finestra_1 = X[:, 0:344, :]      # Da 0 a 344
    finestra_2 = X[:, 328:672, :]    # Da 328 a 672
    finestra_3 = X[:, 656:1000, :]   # Da 656 a 1000
    
    # 2. Impiliamo le finestre creando una nuova dimensione: shape diventerà (N, 3, 344, 1)
    X_stacked = np.stack((finestra_1, finestra_2, finestra_3), axis=1)
    
    # 3. Appiattiamo le prime due dimensioni per ottenere (N*3, 344, 1)
    # In questo modo l'ordine sarà: Paziente1_F1, Paziente1_F2, Paziente1_F3, Paziente2_F1...
    X_new = X_stacked.reshape(-1, 344, 1)
    
    # 4. Duplichiamo le etichette (labels) per farle combaciare col numero di finestre.
    # np.repeat con axis=0 funziona sia per label intere che per one-hot encoding
    y_new = np.repeat(y, 3, axis=0)
    
    return X_new, y_new

# --- ESECUZIONE SUI TUOI DATI ---

# Sostituisci X_train, y_train ecc. con i veri nomi delle tue variabili
X_train_344, y_train_344 = segmenta_dataset(X_train, Y_train)
X_val_344, y_val_344 = segmenta_dataset(X_val, Y_val)
X_test_344, y_test_344 = segmenta_dataset(X_test, Y_test)

print("Nuove shape di Train:", X_train_344.shape, "-", len(y_train_344))
print("Nuove shape di Val:  ", X_val_344.shape, "-", len(y_val_344))
print("Nuove shape di Test: ", X_test_344.shape, "-", len(y_test_344))

In [ ]:
# ==========================================
# 3. TECNICHE DI DATA AUGMENTATION (DA)
# ==========================================
# Nel paper si utilizza spesso l'aggiunta di rumore gaussiano e il drift della linea di base
# Implementiamo un layer di DA nativo di TensorFlow
def ecg_augmentation(signal, label):
    # Aggiunta di rumore Gaussiano (simula artefatti muscolari e termici)
    noise = tf.random.normal(shape=tf.shape(signal), mean=0.0, stddev=0.05, dtype=tf.float32)
    signal = signal + noise
    
    # Random Scaling (simula differenze di impedenza cutanea)
    scale = tf.random.uniform(shape=[1], minval=0.9, maxval=1.1, dtype=tf.float32)
    signal = signal * scale
    return signal, label

batch_size = 64
train_dataset = tf.data.Dataset.from_tensor_slices((X_train_344, y_train_344))
train_dataset = train_dataset.shuffle(buffer_size=1024).map(ecg_augmentation, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val_344, y_val_344)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test_344, y_test_344)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
# ==========================================
# 4. ARCHITETTURA (CNN Leggera < 100k parametri)
# ==========================================
# Architettura a 8 layer convoluzionali come descritta nel paper
def build_lightweight_cnn_binary(input_shape=(344, 1)):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    
    filters = [16, 16, 32, 32, 64, 64, 64, 64]
    kernels = [7, 7, 5, 5, 5, 3, 3, 3]
    
    for f, k in zip(filters, kernels):
        x = layers.Conv1D(filters=f, kernel_size=k, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling1D(pool_size=2)(x)
        
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.5)(x) 
    
    # MODIFICA QUI: un singolo neurone di output con attivazione sigmoide
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

model = build_lightweight_cnn_binary(input_shape=(344, 1))
model.summary() # Verificherai che i parametri sono ~65.000, in linea con "< 100.000"

In [ ]:
# ==========================================
# 5. OTTIMIZZATORE, METRICHE E CALLBACKS
# ==========================================
# Ottimizzatore Adam standard per serie temporali
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# Loss BinaryCrossentropy per classificazione multi-label
model.compile(optimizer=optimizer, 
              loss='binary_crossentropy', 
              metrics=[tf.keras.metrics.AUC(name='auc', multi_label=True), 
                       tf.keras.metrics.BinaryAccuracy(name='accuracy')])

# Tecniche di diminuzione del learning rate e salvataggio pesi
callbacks_list = [
    # Riduce il LR se la loss in validation non migliora per 5 epoche
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1),
    # Ferma l'allenamento in anticipo se la rete va in overfitting
    callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    # Salva il modello finale e i migliori pesi (best_model)
    callbacks.ModelCheckpoint(filepath='./model/best_ptbxl_d1_model_NORM_vs_ALL_344.keras', monitor='val_loss', save_best_only=True, verbose=1)
]

In [ ]:
# ==========================================
# 6. TRAINING
# ==========================================
# Le epoche proposte nel paper o in letteratura per questa architettura variano, 50-70 è lo standard.
EPOCHS = 200 

print("Inizio fase di training...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks_list
)

# Salvataggio Esplicito alla fine dell'addestramento
model.save('./model/final_ptbxl_d1_model_NORM_vs_ALL_344.keras')
model.save_weights('./weights/final_ptbxl_d1_weights_NORM_vs_ALL_344.weights.h5')
print("Modello e pesi salvati con successo.")

In [ ]:
# ==========================================
# 7. VALUTAZIONE E GRAFICI (LOSS E ACCURACY)
# ==========================================
# Test finale sul Fold 10
test_loss, test_auc, test_acc = model.evaluate(test_dataset)
print(f"Risultati sul Test Set (Fold 10) -> Loss: {test_loss:.4f} - AUC: {test_auc:.4f} - Accuracy: {test_acc:.4f}")

# Funzione per plottare Loss e Accuracy (AUC e Binary Accuracy)
def plot_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Grafico della Loss
    ax1.plot(history.history['loss'], label='Train Loss', color='blue')
    ax1.plot(history.history['val_loss'], label='Val Loss', color='orange')
    ax1.set_title('Model Loss (Binary Crossentropy)')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)
    
    # Grafico dell'AUC
    ax2.plot(history.history['auc'], label='Train AUC', color='green')
    ax2.plot(history.history['val_auc'], label='Val AUC', color='red')
    ax2.set_title('Model AUC (Multi-Label)')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('AUC')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig('./img/training_curves_344.png', dpi=300)
    plt.show()

plot_history(history)